# Reason or Recall? — Kaggle runner

Thin entry point: the code lives on GitHub ([umardrazbhatti-work/ReasonOrRecall](https://github.com/umardrazbhatti-work/ReasonOrRecall)); this notebook clones it and calls it. Every experiment goes through the registry-backed runner, so results land in `runs/results.jsonl` and finished experiments are never repeated.

**One-time setup (notebook editor, right-hand sidebar):**
1. **Settings → Accelerator:** `GPU T4 x2` (not P100: it lacks the 4-bit NF4 kernels QLoRA needs). **Settings → Internet:** on.
2. **Input → Add Input → Datasets → Your Datasets:** attach `ror-data` (the uploaded `ror-data.zip`).
3. **Add-ons → Secrets:** add a secret named `HF_TOKEN` (your Hugging Face token) and tick it for this notebook. Needed for gated models (Llama-3.1). Never paste the token into a cell.
4. **Continuing earlier work:** Input → Add Input → *Your Work* → this notebook → attach its latest version's output. Its `runs/` is restored below, so completed experiments are skipped and interrupted training resumes from its checkpoint.

**Run:** pick the `SUITE` in `CONFIG` below, then **Save Version → Save & Run All (Commit)**. Commit mode keeps running after you close the browser and saves `/kaggle/working` as output.
Which suite to run is set by the roadmap (`docs/ROADMAP.md`; the first cell below prints where the project stands):
- `suite_pilot.yaml`: roadmap P1.4, the throughput pilot (~1-1.5 h): the same A5 slice under four batch / memory settings.
- `suite_smoke.yaml`: A5 on 64 training / 32 test items (~10 min), the end-to-end check (roadmap P1.10).
- the phase suites (P3 onward): long runs that pause cleanly before the session limit and resume in the next session (setup step 4).

Set `DRY_RUN = True` to only print the plan.

**After the run:** the last cell prints a plain verdict (completed / failed and why) and writes **one zip**, `ror-output_<runs>_<time>.zip`, to the Output tab. Download it and extract it into `Results/<folder>/` in the local project; it holds the figures and a one-page report (`<runs>/report/index.html`), the results, predictions, per-run logs, the full session log and `requirements.lock`.

In [ ]:
# ---- CONFIG -----------------------------------------------------------------
REPO_URL  = "https://github.com/umardrazbhatti-work/ReasonOrRecall.git"
GIT_REF   = "main"          # branch, tag or commit SHA (pin a SHA to reproduce a run exactly)
SUITE     = "configs/suite_pilot.yaml"      # roadmap P1.4: throughput pilot (~1-1.5 h)
# SUITE   = "configs/suite_smoke.yaml"      # roadmap P1.10: ~10-minute end-to-end check
# SUITE   = "configs/suite_phase1.yaml"     # the study (registry in runs/)
ONLY_ARM  = "A5"            # e.g. "A5"; None = every arm in the suite
MODEL     = "qwen2.5-3b"    # e.g. "qwen2.5-3b"; None = every model in the suite
SPLIT     = "standard"      # "standard" | "clean" | None (both)
SEED      = 0               # e.g. 0; None = every seed
PRIORITY  = None            # roadmap priorities to run, e.g. "M" or "MS"; None = all
DRY_RUN   = False           # True: only print the plan. False: run the experiments.
RUN_TESTS = True            # run pytest (incl. a tiny CPU training run) before anything else
SESSION_HOURS = 11.5        # Kaggle kills sessions at 12h; runs pause cleanly before this

REPO_DIR  = "/tmp/ReasonOrRecall"      # code checkout (not saved as output)

In [ ]:
# ---- bootstrap: fetch the code at GIT_REF and install it ---------------------
import os, re, subprocess, sys, time

SESSION_START = time.time()
SESSION_LOG = "/kaggle/working/session_log.txt"     # everything below, packed into the zip
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["ROR_DEADLINE_UNIX"] = str(SESSION_START + SESSION_HOURS * 3600)

def note(line=""):
    # Print a line and append it to the session log.
    print(line, flush=True)
    with open(SESSION_LOG, "a", encoding="utf-8") as f:
        f.write(f"{line}\n")

# a line that is only a progress-bar redraw (tqdm); log text glued after a bar is kept
BAR = re.compile(r"\d+%\|[^|]*\|[^\[\]]*\[[^\[\]]*\]\s*$")
_last_bar = [0.0]

def sh(cmd, check=True):
    # Stream a shell command's output into this cell and the session log.
    # Progress bars are shown at most once a minute (plus when they finish).
    note(f"$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        line = line.rstrip("\n")
        if BAR.search(line) and "100%|" not in line:
            if time.time() - _last_bar[0] < 60:
                continue
            _last_bar[0] = time.time()
        note(line)
    rc = p.wait()
    if check and rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} fetch --quiet origin {GIT_REF}")
sh(f"git -C {REPO_DIR} checkout --quiet --force FETCH_HEAD")
sh(f"git -C {REPO_DIR} log -1 --oneline")

sh(f"pip install -q -e {REPO_DIR}")
# pinned to the versions the code is tested against (Kaggle image, 2026-09)
sh("pip install -q peft==0.19.1 trl==1.14.0 bitsandbytes==0.50.2")
sh("pip freeze > /kaggle/working/requirements.lock")   # exact versions, saved as output

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

In [ ]:
# ---- environment + secrets ---------------------------------------------------
from ror import kaggle

sh("python -u scripts/roadmap.py")                 # where the project stands (docs/ROADMAP.md)
note(kaggle.gpu_summary())
RUNS_DIR = str(kaggle.runs_dir_for(SUITE))         # e.g. /kaggle/working/runs_smoke
note(f"suite: {SUITE} -> registry + results in {RUNS_DIR}")
found = kaggle.load_secrets(["HF_TOKEN"])          # returns names only, never values
note(f"secrets available: {found or 'none'}")
if "HF_TOKEN" not in found:
    note("note: HF_TOKEN is not attached; gated models (Llama-3.1) will fail to download")

In [ ]:
# ---- data: the attached ror-data dataset -------------------------------------
import json
from pathlib import Path

DATA_DIR = kaggle.ensure_data_dir()
if DATA_DIR is None:
    note("ror-data dataset not attached; building it from the pinned upstream sources instead")
    sh("python -u scripts/prepare_data.py --data-dir /tmp/ror-data --no-zip")
    DATA_DIR = Path("/tmp/ror-data")
os.environ["ROR_DATA_DIR"] = str(DATA_DIR)          # inherited by every script below

manifest = json.loads((DATA_DIR / "ror_data_manifest.json").read_text())
note(f"data: {DATA_DIR}  (preprocess v{manifest['preprocess_version']}, "
     f"built by code {manifest.get('code_commit')})")
for ds, info in manifest["datasets"].items():
    note(f"  {ds:10s} " + ", ".join(f"{s}={v['n']}" for s, v in info["splits"].items()))
note("  clean set: " + ("present" if (DATA_DIR / "clean_set" / "clean.jsonl").exists()
     else "not built yet (clean-split experiments are skipped until it exists)"))

In [ ]:
# ---- restore the registry from earlier versions of this notebook -------------
report = kaggle.restore_runs(Path(RUNS_DIR))
note(f"restored from: {report['sources'] or 'nothing attached (fresh start)'}")
note(f"files copied: {report['files_copied']}, result rows added: {report['results_added']}")
if report["interrupted"]:
    note(f"interrupted runs (resume from checkpoint if attempts remain): {report['interrupted']}")
sh(f"python -u scripts/status.py --runs-dir {RUNS_DIR}")

In [ ]:
# ---- tests -------------------------------------------------------------------
# A failure does not stop the notebook: the experiments are skipped, but the
# verdict and the output zip below are still written.
TESTS_OK = True
if RUN_TESTS:
    TESTS_OK = sh("python -m pytest -q -p no:cacheprovider", check=False) == 0
    note("tests passed" if TESTS_OK else "TESTS FAILED: the experiments will not run")

In [ ]:
# ---- plan (always a dry run) -------------------------------------------------
FILTERS = " ".join(f"--{flag} {value}" for flag, value in
                   [("only", ONLY_ARM), ("model", MODEL), ("split", SPLIT), ("seed", SEED),
                    ("priority", PRIORITY)]
                   if value is not None)
sh(f"python -u scripts/run_suite.py {SUITE} --runs-dir {RUNS_DIR} {FILTERS} --dry-run")

In [ ]:
# ---- run ---------------------------------------------------------------------
if DRY_RUN:
    note("DRY_RUN = True: nothing executed. Set DRY_RUN = False in CONFIG to run the plan above.")
elif not TESTS_OK:
    note("skipped: tests failed (no experiment attempt used). See the test output above.")
else:
    sh(f"python -u scripts/run_suite.py {SUITE} --runs-dir {RUNS_DIR} {FILTERS}", check=False)

In [ ]:
# ---- results -----------------------------------------------------------------
from IPython.display import Image, Markdown, display

sh(f"python -u scripts/status.py --runs-dir {RUNS_DIR}")
sh(f"python -u scripts/aggregate_results.py --runs-dir {RUNS_DIR}")
note("\nlatest results:")
for line in kaggle.latest_results(Path(RUNS_DIR)) or ["(none yet)"]:
    note("  " + line)
table = Path(RUNS_DIR) / "ablation_table.md"
if table.exists():
    display(Markdown(table.read_text()))

# figures: <runs>/report/NN_*.png + index.html (also inside the zip below)
sh(f"python -u scripts/make_report.py --runs-dir {RUNS_DIR}", check=False)
for png in sorted((Path(RUNS_DIR) / "report").glob("*.png")):
    display(Image(filename=str(png)))

In [ ]:
# ---- verdict + the one zip to download ---------------------------------------
verdict = kaggle.session_verdict(Path(RUNS_DIR), since=SESSION_START)
note("\n" + "=" * 30 + " THIS SESSION " + "=" * 30)
if not TESTS_OK:
    note("TESTS FAILED: no experiment was run (no attempt used). The zip has the test output.")
for line in verdict or ["no experiment ran this session (all finished already, "
                        "skipped as not ready, or DRY_RUN)"]:
    note(line)
note(f"session time: {(time.time() - SESSION_START) / 60:.1f} min")
ZIP = kaggle.pack_outputs(Path(RUNS_DIR), extra_files=[
    Path(SESSION_LOG), Path("/kaggle/working/requirements.lock")])
note(f"\nDOWNLOAD: Output tab -> {ZIP.name} ({ZIP.stat().st_size / 1e6:.1f} MB)")
note("extract it into Results/<date>_<suite>_v<version>/ in the local project;")
note(f"open {Path(RUNS_DIR).name}/report/index.html in it for the figures and tables")

## What the output contains

**Download only `ror-output_<runs>_<time>.zip`** and extract it into `Results/<folder>/` locally. It holds `session_log.txt` (every command's output), `requirements.lock` (the exact library versions of the session: for reproducibility, not a result), and the runs folder below without the adapter weights. Open `<runs>/report/index.html` first: the figures (training curve, answer vs gold, error types, accuracy by question type, time and compute; across experiments the ablation, frontier, contamination gap and faithfulness) and every number in tables. The weights stay in the version's output, where the next session restores them from.

In the version's **Output** tab, under `runs_smoke/` (smoke) or `runs/` (study):
- `results.jsonl`: one row per finished experiment with every metric, the full config, git commit, wall time, FLOPs, and training stats (loss, tokens/s).
- `ablation_table.md` / `.csv`, `frontier.csv`: the aggregated study.
- `<exp_id>/`: `predictions.jsonl` (every test item: gold, prediction, correct?, raw model text), `train_stats.json`, `run.log`, `adapter/` (the trained LoRA weights), `status.json`.

## Saving and resuming

- A committed version's output is everything in `/kaggle/working`: the runs folder (registry, `results.jsonl`, per-run logs, adapters, checkpoints) and `requirements.lock`.
- **Long runs:** training checkpoints every 50 steps. Shortly before `SESSION_HOURS` the run saves, stops and is marked *paused* (no attempt used). Commit the next version with this one's output attached and it continues from the checkpoint.
- **Next session:** attach that output as an input (setup step 4) before running. Completed experiments are skipped. A run that was killed mid-way is marked *failed* and retried, resuming training from its latest checkpoint; each retry uses one of its `max_attempts` (2 by default).
- A run that stops because code is still a stub (`NotImplementedError` raised inside `ror/`) does **not** use up an attempt; it is simply picked up again once the code exists.
- Commit `requirements.lock` to the repo once the environment is stable (CLAUDE.md section 7).